# DAAN 545 — Outliers and dimensionality reduction

**Goal:** See whether PCA and t-SNE change if outlier months are removed before the fit.

**Input:** `data/processed/airline_delay_cause_with_airports.csv`. Same months and same six features as `notebooks/06_dimensionality_reduction.ipynb`.

**Approach:**
1. Fit **PCA** and **t-SNE** on every scored month (the published reduction).
2. Refit after the same three removals used in notebook 05. Each refit re-standardizes the rows it keeps.
3. Compare variance shares, sign-aligned loadings, trustworthiness, and where the flags sit in the retained map.

Component signs are arbitrary. Loadings from a refit are flipped so they point the same way as the retained components.

Write-up language and the numbers from the saved run: `docs/notebook_notes.md`.


## 0. Setup

Helpers live in `src/dimred_outlier_impact.py`. The flight-count cutoff is the one already used for clustering and the published reduction. It is not recomputed after a removal. t-SNE uses perplexity 30 and `random_state=42`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dimred_outlier_impact import run_dimred_outlier_impact, save_dimred_outlier_figures

DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
print(f"Project root: {ROOT}")


## 1. What is removed

| Scenario | Months dropped |
|---|---|
| **retain** | None. This is the published PCA / t-SNE fit. |
| **drop consensus** | Flagged by at least two of IQR rates, k-NN, and isolation forest. |
| **drop delay IQR** | `pct_delayed` outside the Tukey fence only. |
| **drop any flag** | Flagged by any one of the three outlier methods. |

`cosine_pc1` through `cosine_pc3` measure how close each refit component is to the retained component after the sign is aligned. A value near 1 means the same axis.


In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "airline_delay_cause_with_airports.csv")
result = run_dimred_outlier_impact(panel)

print("Scored months:", result["meta"]["n_rows"])
print("Features:", result["features"])
print(f"t-SNE perplexity: {result['meta']['tsne_perplexity']}")
display(result["summary"])
print("Share of each component's sum of squares, on the retained PCA:")
display(result["leverage"])


## 2. Variance and loadings

The variance table is the share of scaled variance in each component. The match table records which refit component lined up with each retained component, and whether its sign had to be flipped.


In [ ]:
display(result["variance"])
display(result["loading_match"])
display(result["loadings_aligned"])


## 3. Figures and exports

The variance curves and the loading bars compare fits. The scatter is the **retained** map only: it shows where the flagged months sit, which is why a removal can change a component's sum of squares without rotating the axis.


In [ ]:
paths = save_dimred_outlier_figures(result, FIG_DIR)
for path in paths:
    print(path)

result["summary"].to_csv(DATA_PROCESSED / "dimred_outlier_summary.csv", index=False)
result["variance"].to_csv(DATA_PROCESSED / "dimred_outlier_variance.csv", index=False)
result["loadings_aligned"].to_csv(DATA_PROCESSED / "dimred_outlier_loadings.csv", index=False)
result["loading_match"].to_csv(DATA_PROCESSED / "dimred_outlier_loading_match.csv", index=False)
result["leverage"].to_csv(DATA_PROCESSED / "dimred_outlier_leverage.csv", index=False)
print("Saved dimensionality-reduction outlier tables under data/processed/")
